# 18 — Fitted Bond Curves

Parametric yield-curve models:
- **Nelson-Siegel** (4 params)
- **Svensson** (6 params)
- **Exponential Splines** (5 params)

Plus: curve calibration via AD-based optimization.

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.termstructures.yield_.fitted_bond_curve import (
    NelsonSiegel, Svensson, ExponentialSplines)

---
## 1. Nelson-Siegel

In [ ]:
# β0=6%, β1=−3%, β2=2%, τ=1.5
ns = NelsonSiegel([0.06, -0.03, 0.02, 1.5])

tenors = jnp.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30], dtype=jnp.float64)
ns_zeros = [float(ns.zero_rate(t)) * 100 for t in tenors]
ns_dfs   = [float(ns.discount(t)) for t in tenors]

print(f"{'Tenor':<8s} {'Zero (%)':<10s} {'DF':>10s}")
print("-" * 30)
for t, z, d in zip(tenors, ns_zeros, ns_dfs):
    print(f"{float(t):<8.2f} {z:<10.4f} {d:>10.6f}")

---
## 2. Svensson

In [ ]:
sv = Svensson([0.06, -0.03, 0.02, 0.01, 1.5, 5.0])

sv_zeros = [float(sv.zero_rate(t)) * 100 for t in tenors]

print(f"{'Tenor':<8s} {'NS (%)':<10s} {'SV (%)':<10s} {'Diff (bp)':<10s}")
print("-" * 40)
for t, nz, sz in zip(tenors, ns_zeros, sv_zeros):
    print(f"{float(t):<8.2f} {nz:<10.4f} {sz:<10.4f} {(sz-nz)*100:<10.2f}")

---
## 3. Exponential Splines

In [ ]:
es = ExponentialSplines([1.0, 0.01, -0.005, 0.002, -0.001], kappa=0.05)

es_zeros = [float(es.zero_rate(t)) * 100 for t in tenors]

for t, z in zip(tenors, es_zeros):
    print(f"  {float(t):5.2f}Y : {z:.4f}%")

---
## 4. Visual Comparison

In [ ]:
import matplotlib.pyplot as plt

t_fine = jnp.linspace(0.25, 30, 200)
ns_fine = [float(ns.zero_rate(t)) * 100 for t in t_fine]
sv_fine = [float(sv.zero_rate(t)) * 100 for t in t_fine]
es_fine = [float(es.zero_rate(t)) * 100 for t in t_fine]

plt.figure(figsize=(10, 6))
plt.plot(np.array(t_fine), ns_fine, 'b-', linewidth=2, label='Nelson-Siegel')
plt.plot(np.array(t_fine), sv_fine, 'r--', linewidth=2, label='Svensson')
plt.plot(np.array(t_fine), es_fine, 'g-.', linewidth=2, label='Exp. Splines')
plt.xlabel('Maturity (years)')
plt.ylabel('Zero Rate (%)')
plt.title('Fitted Yield Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. AD: Parameter Sensitivities

In [ ]:
# Jacobian: dZero/dParams for Nelson-Siegel
def ns_zero_fn(params, t):
    model = NelsonSiegel(params)
    return model.zero_rate(t)

params0 = jnp.array([0.06, -0.03, 0.02, 1.5])
maturities = jnp.array([1.0, 5.0, 10.0, 30.0])

# Jacobian: each row = dZ(t_i)/d(params)
jac_fn = jax.jacrev(lambda p: jnp.array([ns_zero_fn(p, t) for t in maturities]))
jac = np.array(jac_fn(params0))

from notebooks._common import plot_heatmap

plot_heatmap(jac,
    xlabels=['β0', 'β1', 'β2', 'τ'],
    ylabels=['1Y', '5Y', '10Y', '30Y'])
plt.title('∂Zero/∂Params (Nelson-Siegel)')
plt.show()

---
## 6. Curve Calibration via AD

Fit Nelson-Siegel to "market" zero rates using gradient descent.

In [ ]:
# Target: rates from NS with known params
true_params = jnp.array([0.06, -0.03, 0.02, 1.5])
cal_tenors  = jnp.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
target_rates = jnp.array([float(NelsonSiegel(true_params).zero_rate(t)) for t in cal_tenors])

# Loss: sum of squared errors
def loss(params):
    model = NelsonSiegel(params)
    fitted = jnp.array([model.zero_rate(t) for t in cal_tenors])
    return jnp.sum((fitted - target_rates) ** 2)

grad_loss = jax.jit(jax.grad(loss))

# Simple gradient descent
params = jnp.array([0.05, 0.0, 0.0, 2.0])  # initial guess
lr = 1.0
losses = []
for i in range(200):
    g = grad_loss(params)
    params = params - lr * g
    losses.append(float(loss(params)))

print(f"True   params : {np.array(true_params)}")
print(f"Fitted params : {np.array(params)}")
print(f"Final loss    : {losses[-1]:.2e}")

plt.figure(figsize=(8, 4))
plt.semilogy(losses)
plt.xlabel('Iteration'); plt.ylabel('Loss')
plt.title('NS Calibration: Loss Convergence')
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. Exercises

1. **Svensson calibration**: Fit the 6-parameter Svensson model with a second hump at 5Y.
2. **Forward rates**: Derive and plot the instantaneous forward rate from each model using AD: $f(t) = -\frac{d}{dt}\ln P(t)$.
3. **Bootstrap vs parametric**: Compare a bootstrapped piecewise curve to a fitted Nelson-Siegel, using the same market data.

---
*End of Notebook 18*